# **Deprecated**
## Already used in [blind_test_interface.ipynb](./blind_test_inference.ipynb)
This module is used for restructure the blind test wav from blind_test_interface.ipynb and shuffle model with random letters 

In [ ]:
import os
import random
import string
import shutil
import json

# Model variables
MODELS = [
    {'model': ['best_model_160217', 'checkpoint_453866'], 'model_path': './model/tsync2_cv_1'},
    {'model': ['best_model_62921', 'best_model_277421', 'checkpoint_240000', 'checkpoint_400000'], 'model_path': './model/tsync2_cv_2'},
]

# Reference names
REFERENCE_FILENAMES = ['cv049_028_mic1.flac',
                       'cv017_012_mic1.flac',
                       'Jinny-04-en.m4a',
                       'Ming-12-th.wav',
                       'ajred_cut.wav',
                       'ekapol_cut.wav']

test = "test4"

all_model = [model['model'] for model in MODELS]
all_model = [model_name for model in all_model for model_name in model]

# Function to get shuffled letters
def get_shuffled_alphabet():
    letters = list(string.ascii_lowercase[:len(all_model)])
    random.shuffle(letters)
    return letters

MODELS = [ {'model':m['model'], 'model_path':m['model_path'].lstrip('./model/')} for m in MODELS]

REFERENCE_FILENAMES = [i.split('.')[0] for i in REFERENCE_FILENAMES]


script_mapping = {}

# Get shuffled alphabet for model representation
shuffled_letters = get_shuffled_alphabet()
all_models_names = [model['model_path'] + '/' + model_name for model in MODELS for model_name in model['model']]
model_letter_map = {model: shuffled_letters[i] for i, model in enumerate(all_models_names)}

# Define base output path for inference files
input_base_path = f"./output/{test}"
output_base_path = f"./output/blindtest/{test}/"
os.makedirs(output_base_path, exist_ok=True)

# Initialize a mapping dictionary
ref_mapping = {}

# Script counter for numbering the scripts
script_counter = 1

ref_id = 0

# Process each reference name and its corresponding model directories
for ref_name in REFERENCE_FILENAMES:
    ref_id+=1
    ref_mapping[ref_id] = ref_name
    for model_name in all_models_names:
        model_letter = model_letter_map[model_name]

        # Construct the model directory path
        model_set_path = os.path.join(input_base_path, ref_name, model_name)

        if not os.path.exists(model_set_path):
            continue

        inference_files = os.listdir(model_set_path)
        script_list = []
        for script_num, file in enumerate(inference_files, start=1):
            if not file.endswith('.wav'):
                continue
            
            if file.rstrip('.wav').lstrip(model_name + '_') not in script_mapping:
                script_mapping[file.rstrip('.wav').lstrip(model_name + '_')] = script_num
            else:
                script_num = script_mapping[file.rstrip('.wav').lstrip(model_name + '_')]
            
            # Create new filename based on the specified format
            new_filename = f"{model_letter}_{ref_id}_{script_num}.wav"
            old_path = os.path.join(model_set_path, file)
            new_path = os.path.join(output_base_path, new_filename)
            
            # Rename the file
            shutil.copyfile(old_path, new_path)

# Save mapping to a text file
with open(os.path.join(output_base_path, 'mapping.txt'), 'w', encoding='utf-8') as f:
    for new_name, old_name in model_letter_map.items():
        f.write(f"{new_name} -> {old_name}\n")
    for ref_id, ref_name in ref_mapping.items():
        f.write(f"{ref_name} -> {ref_id}\n")
    for script_name, script_num in script_mapping.items():
        f.write(f"{script_name} -> {script_num}\n")

# Save the additional JSON file

test_data = {test:[
    {
        "model": f"Model {model_letter.upper()}",
        "modelId": model_name,
        "speakers": [
            {
                "speaker": f"Speaker {ref_id}",
                "scripts": [
                    {"script": f"Script {script_num}", "wav": new_filename}
                    for script_num, new_filename in enumerate(
                        [f"{model_letter}_{ref_id}_{i}.wav" for i in range(1, len(script_mapping)+1)], start=1
                    )
                ]
            }
            for ref_id in range(1, len(REFERENCE_FILENAMES)+1)
        ]
    }
    for model_name, model_letter in model_letter_map.items()
]}

test_data[test] = sorted(test_data[test], key=lambda x: x['model'])

with open(os.path.join(output_base_path, 'data.json'), 'w', encoding='utf-8') as json_file:
    json.dump(test_data, json_file, ensure_ascii=False, indent=4)

print("Renaming complete! Mapping saved.")

Renaming complete! Mapping saved.
